In [11]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
RAW_PATH = ROOT / "data" / "raw" / "cars.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUT_PATH = PROCESSED_DIR / "cars_cleaned.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(RAW_PATH, low_memory=False)
initial_len = len(df)


In [12]:
# 1. ĐỌC DỮ LIỆU
df = pd.read_csv('../data/raw/cars.csv')

COL_PRICE = 'price' 
COL_ODO = 'mileage_v2'      
COL_LOCATION = 'region_name'
COL_TEXT = 'subject'

In [13]:
# 2. CHUẨN HÓA GIÁ & ODO
df[COL_PRICE] = pd.to_numeric(df[COL_PRICE], errors='coerce')
df['price_million'] = df[COL_PRICE] / 1000000
df[COL_ODO] = pd.to_numeric(df[COL_ODO], errors='coerce')
df['price_million'] = df['price_million'].fillna(df['price_million'].median())
df[COL_ODO] = df[COL_ODO].fillna(df[COL_ODO].median())
df[COL_ODO] = df[COL_ODO].astype(int)

In [14]:
# 3. MAP 63 TỈNH THÀNH
def clean_location(val):
    if pd.isna(val):
        return 'Không xác định'
    val = str(val).strip()
    for prefix in ['TP ', 'Tp ', 'Thành phố ', 'Tỉnh ', 'TP. ', 'Tp. ']:
        if val.lower().startswith(prefix.lower()):
            val = val[len(prefix):].strip()     
    mapping = {
        'Hồ Chí Minh': 'Hồ Chí Minh', 'Hcm': 'Hồ Chí Minh',
        'Hà Nội': 'Hà Nội', 'Hn': 'Hà Nội',
        'Bria - Vũng Tàu': 'Bà Rịa - Vũng Tàu', 'Thừa Thiên Huế': 'Thừa Thiên - Huế'}
    return mapping.get(val, val)
df[COL_LOCATION] = df[COL_LOCATION].apply(clean_location)

In [15]:
# 4. TẠO CỜ NHỊ PHÂN NLP TỪ TEXT
text_series = df[COL_TEXT].astype(str).str.lower()
df['flag_chinh_chu'] = text_series.str.contains(r'chính chủ|1 chủ|một chủ|gia đình', regex=True).astype(int)
df['flag_bao_duong_hang'] = text_series.str.contains(r'bảo dưỡng hãng|lịch sử hãng|full lịch sử|bảo hành', regex=True).astype(int)
# Lọc tin spam, tin tìm mua, tin cho thuê
spam_pattern = r'cho thuê|thuê xe|tìm mua|cần mua|thu mua|góp xe|đặt cọc'
spam_mask = df['subject'].astype(str).str.contains(spam_pattern, case=False, regex=True)
df = df[~spam_mask]

# Lọc các mức giá không thực tế đối với ô tô lưu thông
df = df[df['price_million'] >= 20.0]


In [16]:
# 5. LỌC NGOẠI LAI (OUTLIERS) BẰNG IQR
def remove_outliers_iqr(dataframe, columns, min_bounds=None):
    df_out = dataframe.copy()
    for col in columns:
        Q1 = df_out[col].quantile(0.25)
        Q3 = df_out[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        if min_bounds and col in min_bounds:
            lower_bound = max(lower_bound, min_bounds[col])
        df_out = df_out[(df_out[col] >= lower_bound) & (df_out[col] <= upper_bound)]
    return df_out

# Chặn dưới giá xe tối thiểu 20 triệu VNĐ và ODO tối thiểu 0 km
df = remove_outliers_iqr(df, ['price_million', COL_ODO], min_bounds={'price_million': 20.0, COL_ODO: 0})
final_len = len(df)


In [17]:
def safe_map(series, mapping):
    numeric_s = pd.to_numeric(series, errors='coerce')
    full_m = {}
    for k, v in mapping.items():
        full_m[k] = v
        full_m[float(k)] = v
        full_m[str(k)] = v
        full_m[f"{float(k):.1f}"] = v
    mapped = numeric_s.map(full_m)
    return mapped.fillna(series)

gearbox_map = {
    1: 'Số tự động',
    2: 'Số sàn',
    3: 'Bán tự động'
}

fuel_map = {
    1: 'Xăng',
    2: 'Dầu',
    3: 'Hybrid',
    4: 'Điện'
}

cartype_map = {
    1: 'Sedan',
    2: 'Coupe',
    3: 'SUV / Crossover',
    4: 'Hatchback',
    5: 'Minibus / Xe chuyên dụng',
    6: 'Bán tải',
    7: 'Mui trần',
    8: 'MPV',
    9: 'Van / Minivan'
}

carcolor_map = {
    1: 'Đen',
    2: 'Trắng',
    3: 'Bạc',
    4: 'Xám',
    5: 'Đỏ',
    6: 'Xanh dương',
    7: 'Vàng',
    8: 'Cam',
    9: 'Nâu',
    10: 'Xanh lá',
    11: 'Hồng',
    12: 'Màu khác'
}

carorigin_map = {
    1: 'Lắp ráp trong nước',
    2: 'Nhập khẩu Ấn Độ',
    3: 'Nhập khẩu Hàn Quốc',
    4: 'Nhập khẩu Thái Lan',
    5: 'Nhập khẩu Nhật Bản',
    6: 'Nhập khẩu Trung Quốc',
    7: 'Nhập khẩu Indonesia',
    8: 'Nhập khẩu Mỹ',
    9: 'Nhập khẩu Đức',
    10: 'Nhập khẩu Đài Loan'
}

carseats_map = {
    1: '4 chỗ',
    2: '5 chỗ',
    3: '7 chỗ',
    4: '5 chỗ',
    5: '2 chỗ',
    6: '8 chỗ',
    7: '9 chỗ',
    8: '10 chỗ',
    9: '12 chỗ',
    11: '16 chỗ',
    12: '6 chỗ'
}

df['gearbox'] = safe_map(df['gearbox'], gearbox_map)
df['fuel'] = safe_map(df['fuel'], fuel_map)
df['cartype'] = safe_map(df['cartype'], cartype_map)
df['carcolor'] = safe_map(df['carcolor'], carcolor_map)
df['carorigin'] = safe_map(df['carorigin'], carorigin_map)
df['carseats'] = safe_map(df['carseats'], carseats_map)


In [18]:
CURRENT_YEAR = 2026
if 'mfdate' in df.columns:
    df['mfdate'] = pd.to_numeric(df['mfdate'], errors='coerce')
    df['car_age'] = CURRENT_YEAR - df['mfdate']
    df.loc[df['car_age'] < 0, 'car_age'] = 0

cols_to_drop = ['images', 'image', 'source_url']
df = df.drop(columns=cols_to_drop, errors='ignore')

df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')


In [19]:
# 6. KIỂM TRA VÀ LƯU FILE
print('--- KẾT QUẢ LỌC NGOẠI LAI BẰNG IQR ---')
print(f'Số dòng ban đầu: {initial_len}')
print(f'Số dòng sau khi lọc: {final_len}')
print(f'-> Đã loại bỏ: {initial_len - final_len} dòng ngoại lai có giá hoặc ODO quá bất thường.\n')
print('--- KẾT QUẢ TẠO CỜ NLP (5 DÒNG ĐẦU) ---')
display(df[[COL_TEXT, 'flag_chinh_chu', 'flag_bao_duong_hang']].head())
output_file = '../data/processed/cars_cleaned.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f'\nĐã lưu file hoàn chỉnh: {output_file}')


--- KẾT QUẢ LỌC NGOẠI LAI BẰNG IQR ---
Số dòng ban đầu: 15000
Số dòng sau khi lọc: 13520
-> Đã loại bỏ: 1480 dòng ngoại lai có giá hoặc ODO quá bất thường.

--- KẾT QUẢ TẠO CỜ NLP (5 DÒNG ĐẦU) ---


,subject,flag_chinh_chu,flag_bao_duong_hang
0,ISUZU DMAX NHẬP THÁI SỐ TỰ ĐỘNG SIÊU KENG RẤT ĐẸP,0,0
1,Ranger XLS AT,0,0
2,terano ll máy dầu 7 chỗ diezen 2 cầu tubo xe đẹp,0,0
3,BMW X6 2010 Full Đồ Chơi Của Hãng,0,0
4,Toyota Fortuner 4x2 2016 một chủ,1,0



Đã lưu file hoàn chỉnh: ../data/processed/cars_cleaned.csv
